# 2026-10-03 Lab Data Discovery

**Author:** Kyle Shannon · **Notebook:** trajectories-eda · **Created:** 2026-10-03T18:45:52-07:00

**Purpose:** This notebook will load lab data from Dr. Nitz's lab from the Olson 2017 paper. I will then use this as a basis to build a data loader for trajectory data, and then perform a light EDA on the data. Mostly I want to just get a feel for the trajectory and analysis files. This is all very general level work, not directed to answer any question. Following notebooks will work on deriving a correct schema and validating it against the data to ensure we can subset the data for synthetic trajectory creation.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import subprocess
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401  registers the SciencePlots styles

import subiculum_rnn
from subiculum_rnn.environments import list_environments, load_environment
from subiculum_rnn.store import resolve_store

plt.style.use(["science", "notebook", "no-latex"])
rng = np.random.default_rng(0)
store = resolve_store(None)
FIGURES = Path("figures")

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                        text=True).stdout.strip() or "unknown"
print(f"subiculum-rnn {subiculum_rnn.__version__}  commit {commit}  {date.today()}")
print(f"store: {store.root} ({store.source})")
# print(f"store: {store_path}")
print(f"environments: {', '.join(list_environments())}")

## Cracking Open some Plexus recording data

1. scipy's [lib](https://docs.scipy.org/doc/scipy/reference/io.html) has a matlab `.mat` set of tools that can help us extract the data and read these files. Specifically `.whosmat` and `.loadmat`. Scipy docs notes that hdf5 is required for newer matlab file formats, this is the older version (verified in python cli with `.whosmat`)
2. With `.loadmat` we can turn matlab data into numpy data and matlab structs into objects with a `_fieldnames` list.
3.  


### Trajectory DVT File Variable example

In [ ]:
# Trying to read the matlab data

import scipy.io as sio

path = "/Volumes/haruhi/phd/2017_olson_nitz_data/2017_NatNeuro_AxisPaper/allSubData/NS16Data/NS16_rec32_110315001ProcessedDVT.mat"

# "JL1HPCData/JL1_rec23_040315001ProcessedDVT.mat"

def walk(obj, prefix="", depth=0):
    if hasattr(obj, "_fieldnames"):
        for f in obj._fieldnames:walk(getattr(obj, f), f"{prefix}.{f}", depth + 1)
    elif isinstance(obj, np.ndarray):
        print(f"{prefix:45s} {obj.dtype} {obj.shape}")
    else:
        print(f"{prefix:45s} {type(obj).__name__} {obj!r:.40}")

matlab_file = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
for k, v in matlab_file.items():
    if not k.startswith("__"):
        walk(v, k)

### Analysis RateMaps File Variable example

In [ ]:
# MATLAB v7.3 files are HDF5, so loadmat refuses them. Look inside one with h5py.
import h5py

rmaps_path = "/Volumes/haruhi/phd/2017_olson_nitz_data/2017_NatNeuro_AxisPaper/allSubData/JL1HPCData/JL1_rec23_040315001_rmaps.mat"

def walk_h5(path):
    # MATLAB writes column-major, so h5py shapes are the MATLAB dims reversed
    refs = 0
    def show(name, node):
        nonlocal refs
        if name.startswith("#refs#/"):
            refs += 1
            return
        cls = node.attrs.get("MATLAB_class", b"").decode()
        kind = f"{node.dtype} {node.shape}" if isinstance(node, h5py.Dataset) else "group"
        print(f"{name:45s} {cls:10s} {kind}")
    with h5py.File(path, "r") as f:
        f.visititems(show)
    print(f"{refs} objects under #refs# (cell array contents)")

walk_h5(rmaps_path)

In [ ]:
import re
from pathlib import Path
import pandas as pd
import scipy.io as sio


matlab_data_path = "/Volumes/haruhi/phd/2017_olson_nitz_data/2017_NatNeuro_AxisPaper/allSubData/"
ROOT = Path(matlab_data_path)
NAME = re.compile(r"^(?P<subject>[A-Za-z]+\d+)_rec(?P<rec>\d+)_(?P<stamp>\d{6})(?P<session>\d{3})_?(?P<rest>.+)\.mat$")

def parse(path):
    m = NAME.match(path.name)
    if m is None:
        return None
    d = m.groupdict()
    *tags, kind = d.pop("rest").split("_")
    return {"group": path.parent.name, "subject": d["subject"], "rec": int(d["rec"]),
            "date": pd.to_datetime(d["stamp"], format="%m%d%y"), "session": d["session"],"tag": "_".join(tags), "kind": kind, "size_mb": round(path.stat().st_size / 1e6, 1),
            "path": str(path)}

rows, skipped = [], []
for path in sorted(ROOT.rglob("*.mat")):
  info = parse(path)
  if info is None:
    skipped.append(str(path.relative_to(ROOT)))
  else:
    rows.append(info)

catalog = pd.DataFrame(rows)
print(f"{len(catalog)} data files, {len(skipped)} skipped:")
print("\n".join(skipped))
catalog.groupby(["group", "kind", "tag"]).size()

In [ ]:
m = sio.loadmat(path, squeeze_me=True)
raw, proc, hd = m["rawDVT"], m["processedDVT"], m["HDRadians"]
np.set_printoptions(precision=2, suppress=True, linewidth=140)
print(raw[:3]); print(proc[:3])

t = raw[:, 1]
print("timestamp steps:", np.unique(np.round(np.diff(t), 4))[:5], " duration:", t[-1] - t[0])
for name in ("samplesLost", "samplesFilled", "samplesUnfilled"):
    print(f"{name:16s} fraction per light: {m[name].astype(bool).mean(0)}")

In [ ]:
for j in range(proc.shape[1]):
    col = proc[:, j]
    print(f"col {j}: min {np.nanmin(col):9.2f}  max {np.nanmax(col):9.2f}  nan {np.isnan(col).mean():.3f}")

mid = (proc[:, 2:4] + proc[:, 4:6]) / 2
print("cols 6:8 are the LED midpoint:", np.allclose(proc[:, 6:8], mid, equal_nan=True))

In [ ]:
d = proc[:, 4:6] - proc[:, 2:4]
candidates = {
    "led1 -> led2":            np.arctan2(d[:, 1], d[:, 0]),
    "led2 -> led1":            np.arctan2(-d[:, 1], -d[:, 0]),
    "led1 -> led2, y flipped": np.arctan2(-d[:, 1], d[:, 0]),
    "led2 -> led1, y flipped": np.arctan2(d[:, 1], -d[:, 0]),
}
for name, ang in candidates.items():
    err = np.angle(np.exp(1j * (hd - ang)))
    print(f"{name:26s} median |error| {np.degrees(np.nanmedian(np.abs(err))):6.1f} deg")

In [ ]:
vel = m["vel"]
mine = np.diff(np.convolve(proc[:, 2], np.ones(6) / 6, mode="valid")) * 60   # LED1 x, their window, per second
for a, b in [(0, 0), (1, 0), (0, 1), (1, 1)]:print(f"vel[:, {a}, {b}] vs my LED1 vx: corr {np.corrcoef(vel[:, a, b], mine)[0, 1]:.3f}")

In [ ]:
unf = m["samplesUnfilled"].astype(bool)                 # per light
led = proc[:, 2:6].copy()
led[unf[:, 0], 0:2] = np.nan
led[unf[:, 1], 2:4] = np.nan
print("sentinel rows match unfilled flags:",
      np.array_equal((proc[:, 2] == 1) & (proc[:, 3] == 1), unf[:, 0]),
      np.array_equal((proc[:, 4] == 1) & (proc[:, 5] == 1), unf[:, 1]))
led1, led2 = led[:, 0:2], led[:, 2:4]                  # front, back
print("HD missing where back light is unfilled:", np.isnan(hd[unf[:, 1]]).mean())

In [ ]:
pairs = [(6, 2, "6:8 vs LED1"), (6, 4, "6:8 vs LED2"), (8, 2, "8:10 vs LED1"), (8, 4, "8:10 vs LED2")]

for a, b, label in pairs:
    left = proc[:, a:a + 2]
    right = proc[:, b:b + 2]
    same = np.mean(np.all(left == right, axis=1))
    diff = np.nanmedian(np.abs(left - right))
    print(f"{label}: identical rows {same:.3f}, median |diff| {diff:.2f} px")

In [ ]:
vel = m["vel"]
x1 = np.where(unf[:, 0], np.nan, proc[:, 2])
mine = np.diff(np.convolve(x1, np.ones(6) / 6, mode="valid")) * 60
for a, b in [(0, 0), (1, 0), (0, 1), (1, 1)]:
    v = vel[:, a, b]
    ok = np.isfinite(v) & np.isfinite(mine)
    print(f"vel[:, {a}, {b}]  corr {np.corrcoef(v[ok], mine[ok])[0, 1]:.3f}  n={ok.sum()}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax[0].plot(t, led1[:, 0], lw=0.3); ax[0].set_ylabel("x (px)")
ax[1].plot(t, led1[:, 1], lw=0.3); ax[1].set_ylabel("y (px)"); ax[1].set_xlabel("t (s)")

In [ ]:
pos = led1.copy()
missing_front = np.isnan(pos[:, 0])
pos[missing_front] = led2[missing_front]
missing = np.isnan(pos[:, 0])

edges = np.flatnonzero(np.diff(np.r_[0, missing.astype(int), 0]))
starts, ends = edges[::2], edges[1::2]
for s, e in zip(starts, ends):
    if (e - s) / 60 > 10:
        print(f"gap from {t[s]:.1f} s to {t[e - 1]:.1f} s")

In [ ]:
arena = (t > 20) & (t < 295) & ~missing
track = (t > 340) & ~missing

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a, sel, title in [(ax[0], arena, "arena"), (ax[1], track, "track")]:
    a.plot(pos[sel, 0], pos[sel, 1], lw=0.3, color="0.3")
    sc = a.scatter(pos[sel, 0], pos[sel, 1], c=t[sel], s=1, cmap="viridis")
    a.set_aspect("equal"); a.invert_yaxis()
    a.set_title(title); a.set_xlabel("x (px)"); a.set_ylabel("y (px)")
fig.colorbar(sc, ax=ax, label="t (s)", shrink=0.8)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist2d(pos[track, 0], pos[track, 1],
          bins=[np.arange(80, 600, 5), np.arange(40, 440, 5)], cmin=1)
ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_title("track occupancy, 5 px bins")

In [ ]:
y = pos[:, 1]
mid = 250
runs = (y[1:] < mid) & (y[:-1] >= mid) & track[1:]      # y decreases toward the top in video coordinates
print("runs toward the goals:", runs.sum())

top = track & (y < 120)
plt.hist(pos[top, 0], bins=np.arange(80, 600, 5))
plt.title("x while at the top: expect four goal clusters")

In [ ]:
dt = 1 / 60
vel = np.gradient(pos, dt, axis=0)                  # px/s, central differences
hd_clean = np.where(unf[:, 1] | unf[:, 0], np.nan, hd)   # no heading without both lights

table = pd.DataFrame({
    "t": t,
    "x": pos[:, 0],
    "y": pos[:, 1],"vx": vel[:, 0],
    "vy": vel[:, 1],
    "speed": np.hypot(vel[:, 0], vel[:, 1]),
    "head_dir_deg": np.degrees(hd_clean),
    "epoch": np.select([arena, track], ["arena", "track"], default="gap"),
})
table.loc[missing, ["x", "y", "vx", "vy", "speed"]] = np.nan

table.head(10)

In [ ]:
table.groupby("epoch")[["x", "y", "speed", "head_dir_deg"]].describe().round(1).T

In [ ]:
table[table.epoch == "track"].iloc[20000:20010]   # a slice mid-session, mid-run

In [ ]:
table.dropna(subset=["x"]).head(10)                # first tracked frames, arena
table[(table.t > 400) & (table.t < 400.2)]          # a dozen frames mid-run on the track
table.groupby("epoch")[["x", "y", "speed", "head_dir_deg"]].describe().round(1).T